Test with .tf savedmodel format model file

In [ ]:
! pip install ultralytics

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
import cv2, os
import numpy as np
import tensorflow as tf
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt

Loading required models

In [ ]:
fish_model = YOLO('fish.pt')
cuts_damages_model = YOLO("cut-seg.pt")
savedmodel_path = "sardine_model_20epochs_k_gpu/kaggle/working/sardine_model_20epochs_k_gpu"
loaded = tf.saved_model.load(savedmodel_path)
infer = loaded.signatures["serving_default"]
softness_model = infer

In [ ]:
def yolo_results(img_path, model, imgsz=640, conf=0.75, iou=0.7):
    results = model(img_path, imgsz=imgsz, conf=conf, iou=iou, verbose=False)
    return results

cuts and damages utils

In [ ]:
def process_image(image, size=(640, 640)): # for eye model
    h, w = image.shape[:2]
    aspect_ratio = w / h
    if aspect_ratio > 1:
        new_w = size[0]
        new_h = int(new_w / aspect_ratio)
    else:
        new_h = size[1]
        new_w = int(new_h * aspect_ratio)

    resized_image = cv2.resize(image, (new_w, new_h))

    pad_h = (size[1] - new_h) // 2
    pad_w = (size[0] - new_w) // 2
    padded_image = cv2.copyMakeBorder(
        resized_image, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_CONSTANT, value=(255, 255, 255)
    )
    padded_image = cv2.resize(padded_image, (640,640))
    return np.asarray(padded_image)
    
def cuts_damages(seg_img_list):
    for i in range(len(seg_img_list)):
        image = process_image(seg_img_list[i])
        image = image[:,:,::-1]
        results = yolo_results(image, cuts_damages_model, conf=0.5, iou=0.7)
        cuts_detected = len(results[0].boxes.cls)
        if cuts_detected == 0:
            return None
        else:
            return True

In [ ]:
def softness_model_predict(seg_img, model):
    img = cv2.resize(seg_img, (224, 224))
    img_array = tf.keras.applications.densenet.preprocess_input(img)
    img_array = tf.expand_dims(img_array, axis=0)
    labeling = infer(img_array)
    pred = 'Good' if labeling['output_0'].numpy()[0][0] >= 0.5 else 'Bad'    
    return pred 

Drawing bboxes, masks and labels

In [ ]:
def add_text(img, text):
    font = cv2.FONT_HERSHEY_SIMPLEX
    fontScale = 1
    color = (0, 0, 0)
    thickness = 2

    # Using cv2.putText() method
    out_img = cv2.putText(img, text, (10, 30), font, fontScale, color, thickness, cv2.LINE_AA)
    return out_img


def get_segmented_img(pred, white_background=False, index=0) -> list:
    """Get segmented images from prediction result.
    Args:
        pred: Prediction result from YOLO model.
        white_background (bool): If True, set background to white. Default is False.
        index (int): Index of the mask to process. Default is 0.
    Returns:
        list: List of segmented images as numpy arrays.
    """
    img_shape = pred.orig_shape
    img = pred.orig_img.copy()
#     img = cv2.imread(img_path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img_shape[0], img_shape[1]
    masks = pred.masks.xy
    seg_img_list = []
    boxes = []
    for mask in masks:
#         mask = masks[index]
        mask_points = mask.astype(int)
        binary_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(binary_mask, [mask_points], 255)
        masked_img = cv2.bitwise_and(img, img, mask=binary_mask)
        x, y, w, h = cv2.boundingRect(mask_points)
        x1, y1, x2, y2 = (x, y, x+w, y+h)
        box = [x1, y1, x2, y2]
        boxes.append(box)
        segment_image = np.zeros((height, width, 3), dtype=np.uint8)
        segment_image[y:y+h, x:x+w] = masked_img[y:y+h, x:x+w]
        segment_image = segment_image[y:y+h, x:x+w]
        # white_background = False
        if white_background:
            black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
            segment_image[black_mask] = [255, 255, 255]
        seg_img_list.append(segment_image)
    return seg_img_list, masks, boxes

def get_colors(pred_class):
    if pred_class == 'Bad':
        color = (0, 0, 255)
    else:
        color = (0, 255, 0)
    return color

def draw_annotated_boxes(img, box, color, text):
    cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]),
                  color, 5)
    # Define the font and scale of the text
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 4
    font_color = (255, 255, 255)  # White color
    font_thickness = 15
    text_position = (box[0], box[1] - 10)

    # Get the size of the text to calculate the background rectangle
    (text_width, text_height), baseline = cv2.getTextSize(
                                text, font, font_scale, font_thickness)

    # Calculate the position for the background rectangle
    background_rect_start = (box[0], box[1] - text_height - 10)
    # background_rect_end = (box[0] + text_width, box[1] - baseline + 10)
    background_rect_end = (box[0] + text_width, box[1])

    # Draw the background rectangle
    cv2.rectangle(img, background_rect_start,
                  background_rect_end, color, -1)

    # Put the text on the image
    out_img = cv2.putText(img, text, text_position,
                          font, font_scale, font_color, font_thickness)
    return out_img

def fill_mask(img, mask, color):
    height, width = img.shape[0], img.shape[1]
    binary_mask = np.zeros((height, width), dtype=np.uint8)
    mask = mask.astype(np.int32)
    cv2.fillPoly(binary_mask, [mask], 255)
    # return binary_mask

    masked_img = cv2.bitwise_and(img, img, mask=binary_mask)
    masked_img[binary_mask == 255] = color
    transparency = 0.8
    masked_img = cv2.addWeighted(img, 1.0, masked_img, transparency, 0)
    return masked_img

def draw_bbox_masks_labels(img, mask, box, text, color):
    masked_img = fill_mask(img, mask, color)
    out_img = draw_annotated_boxes(masked_img, box, color, text)
    return out_img

getting final predictions for each image

In [ ]:
def get_result_img(results):
    obj_detected = len(results[0].boxes.cls)
    if obj_detected == 0:
        # add text 'No fish detected' over the image
        output_img = add_text(results[0].orig_img, 'No fish detected')
        return output_img
    else:
        output_img = results[0].orig_img
        for i in range(obj_detected):
            seg_img_list, mask, bbox = get_segmented_img(results[0], True, i)
            # Go to Cuts and damages model
            cuts = cuts_damages(seg_img_list)
            if (cuts==None) or (cuts == False):
                seg_img_list, mask, bbox = get_segmented_img(results[0], False, i)
                pred = softness_model_predict(seg_img_list, softness_model)
                color = get_colors(pred)
                text = f'{pred}: Soft'
                if pred == 'Good': text = f'{pred}'
                print(text)
                output_img = draw_bbox_masks_labels(output_img, mask,
                                                    bbox, text, color)
            else:
                pred = 'Bad'
                color = get_colors(pred)
                text = f'{pred}: Cuts'
                print(text)
                output_img = draw_bbox_masks_labels(output_img, mask, bbox,
                                                    text, color)
    return output_img

In [ ]:
folder = "New app testing data/input/2024-06-11/sardine/Bad"
img_path = os.path.join(folder, '2024-06-11_10:00:58_(13522)_sardine_input.jpeg')
results = yolo_results(img_path, fish_model, conf=0.75, iou=0.7)
output_img = get_result_img(results)
rgb_img = cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB)
plt.imshow(rgb_img);